<a href="https://colab.research.google.com/github/Yusef-Hazem/Lang-Frameworks/blob/main/06_Self_Reflection_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Langchain/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Langchain


In [ ]:
#Python 3.12.13
!pip install -U langchain==1.3.11
!pip install requests==2.34.2
!pip install -U langchain_community==0.4.2
!pip install -U python-dotenv==1.2.2
!pip install -U langchain-huggingface==1.2.2
!pip install faiss-cpu==1.14.3
!pip install langchain-groq==1.1.3
!pip install chromadb==1.5.9
!pip install langchain-classic==1.0.8
!pip install -U langgraph==1.2.7

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS ,Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import bs4
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

True

In [ ]:
docs = WebBaseLoader(["https://www.coursera.org/learn/introduction-to-retrieval-augmented-generation-rag" ,
                      "https://aws.amazon.com/what-is/retrieval-augmented-generation/"]).load()

## Retriever Node

In [ ]:
# spliting
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
# embedding modle
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
embedding_model = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs)

#  create vDB
vector_db = Chroma.from_documents(chunks, embedding_model)
retriever = vector_db.as_retriever()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Retrieval Grader Node

In [ ]:
from typing import Literal
from langchain_core.utils.pydantic import BaseModel  ,Field

In [ ]:
#define a class of grade documents
class GradeDocument(BaseModel) :

  score : str = Field(... , description= "Documents are relivant to Query :'yes' or 'no' " )

llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
structred_llm_matching_grader = llm.with_structured_output(GradeDocument)


system = """You are a grader assessing relevance of a retrieved document to a user question. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human","Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)
retrieval_grader =(grade_prompt | structred_llm_matching_grader )

question = "what is rag system "
docs = retriever.invoke(question)
document_to_grade = docs[1].page_content


retrieval_grader.invoke({"document": document_to_grade, "question": question})

GradeDocument(score='yes')

## Generate Node

In [ ]:
llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)

template = "based on this context :\n {context} \n\n answer this question {question}"
prompt = ChatPromptTemplate.from_template(template)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


generate_answer = (prompt | llm | StrOutputParser())
generate_answer.invoke({"context": format_docs(docs), "question": question})

'Based on the provided context, RAG stands for Retrieval-Augmented Generation. It is a type of Artificial Intelligence (AI) technology that combines the capabilities of large language models (LLMs) with retrieval mechanisms to generate more accurate and informative responses.\n\nIn simpler terms, RAG is a system that uses a combination of natural language processing (NLP) and information retrieval techniques to generate human-like text based on a given prompt or input. It works by retrieving relevant information from a database or knowledge base and then using that information to generate a response.\n\nThe key components of a RAG system include:\n\n1. Large Language Models (LLMs): These are AI models that are trained on vast amounts of text data and can generate human-like language.\n2. Retrieval Mechanisms: These are algorithms that retrieve relevant information from a database or knowledge base based on a given prompt or input.\n3. Generation: This is the process of using the retrie

## Hallucination Grader `edge`

In [ ]:
class GradeHallucination(BaseModel):
  score: str = Field(description="Answer is grounded in the facts, 'yes' or 'no'")

llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
structred_llm_hallucination = llm.with_structured_output(GradeHallucination)

system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)
hallucination_grader = hallucination_prompt | structred_llm_hallucination

## Answer Grader `edge`

In [ ]:
class GradeAnswer(BaseModel):
  score: str = Field(description="Answer addresses the question, 'yes' or 'no'")

llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
structred_llm_answer_grader = llm.with_structured_output(GradeAnswer)


system = """You are a grader assessing whether an answer addresses / resolves a question \n
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structred_llm_answer_grader


## Question Re-writer Node


In [ ]:
llm = ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)

system = """You a question re-writer that converts an input question to a better version that is optimized \n
     for vectorstore retrieval. Look at the input and try to reason about the underlying sematic intent / meaning. Just say the question. I dont need any explanation just question is enough"""
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Here is the initial question: \n\n {question} \n Formulate an improved question."),
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()

# Graph

In [ ]:
from typing import List
from typing_extensions import TypedDict
from langchain_core.documents import Document
class GraphState (TypedDict) :
  question : str
  generation : str
  documents : List[Document]

# nodes
#--------------------------------------------------------
def retrive (state) :
  print("------retrive---node-----")
  question = state["question"]

  documents = retriever.invoke(question)
  return {"documents": documents , "question" : question }
#--------------------------------------------------------
def grade_docs (state) :
  print("---------GradeDocs----node----")
  question =state["question"]
  documents = state["documents"]
  retrived_docs =[]
  for doc in documents:
    score= retrieval_grader.invoke({"document": doc , "question" : question })
    grade = score.score

    if grade =="yes" :
      print("---GRADE: DOCUMENT RELEVANT---")
      retrived_docs.append(doc)
    else :
      print("---GRADE: DOCUMENT NOT RELEVANT---")
  return {"documents" : retrived_docs , "question" : question}
  #--------------------------------------------------------
def generate_answer (state) :
  print("---------generate_answer----node----")
  question =state["question"]
  documents = state["documents"]

  generation = generate_answer.invoke({"context": format_docs(documents), "question": question})
  return {"documents" : documents ,"question" : question ,"generation" : generation }

  #-------------------------------------------------------

def re_write_question(state) :

  print("---REWRITE_QUESTION------NODE--")
  question = state["question"]
  documents = state["documents"]

  # Re-write question
  better_question = question_rewriter.invoke({"question": question})
  return {"documents": documents, "question": better_question}




In [ ]:
#Edges

def decide_to_generate (state) :
    print("---ASSESS GRADED DOCUMENTS---")
    question = state["question"]
    filtered_documents = state["documents"]



    if not filtered_documents :
      print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---")
      return "re_write_question"
    else:

    # We have relevant documents, so generate answer
      print("---DECISION: GENERATE---")
      return "generate_answer"



def is_there_hallucination_and_good_answer (state) :
    print("---CHECK HALLUCINATIONS---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]


    score = hallucination_grader.invoke({"documents": documents, "generation": generation})
    grade = score.score

    if grade == "yes" :
      print("-----Check is answer useful --------")
      score = answer_grader.invoke({"question": question,"generation": generation})
      grade = score.score

      if grade == "yes":
        print("---DECISION: GENERATION ADDRESSES QUESTION---")
        return "useful"
      else:
        print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION--- i'm rewriting the question ---")
        return "not useful"
    else:
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"

In [ ]:

from langgraph.graph import StateGraph ,END


workflow = StateGraph(GraphState)

workflow.add_node("retrive", retrive)
workflow.add_node("grade_docs", grade_docs)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("re_write_question", re_write_question )


# build our graph

workflow.set_entry_point("retrive")
workflow.add_edge("retrive", "grade_docs")
workflow.add_conditional_edges("grade_docs" , decide_to_generate,{"re_write_question": "re_write_question",
                                                                       "generate_answer": "generate_answer"})
workflow.add_edge("re_write_question", "retrive")
workflow.add_conditional_edges("generate_answer" ,is_there_hallucination_and_good_answer , { "not supported": "generate_answer",
                                                                                              "useful": END,
                                                                                              "not useful": "re_write_question"} )

app = workflow.compile()

In [ ]:
inputs = {"question": "Explain how the different types of agent memory work?"}
app.stream(inputs)

<generator object Pregel.stream at 0x11c620d0>

In [ ]:

# you can use app.stream and yield the output dict
app.invoke({"question": "what is rag"})["generation"]

------retrive---node-----
---------GradeDocs----node----
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
---------generate_answer----node----
---CHECK HALLUCINATIONS---
-----Check is answer useful --------
---DECISION: GENERATION ADDRESSES QUESTION---


"According to the context, **RAG (Retrieval-Augmented Generation)** is the process of optimizing the output of a large language model by referencing an authoritative knowledge base outside of its training data sources before generating a response. In other words, RAG extends the capabilities of Large Language Models (LLMs) to specific domains or an organization's internal knowledge base, without the need to retrain the model, making it a cost-effective approach to improving LLM output."

In [ ]:
app.invoke({"question": "Explain how the different types of agent memory work?"})["generation"]

------retrive---node-----
---------GradeDocs----node----
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---
---REWRITE_QUESTION------NODE--
------retrive---node-----
---------GradeDocs----node----
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---
---REWRITE_QUESTION------NODE--
------retrive---node-----
---------GradeDocs----node----
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---
---REWRITE_QUESTION------NO

KeyboardInterrupt: 